# Program O-R — David's FULL model laboratory

This notebook gives David freedom to edit the **neural model** while locking the scientific object: Program O-R physics, `Discrete(4)` actions, allowed observations, terminal canonical ReT, development-only seeds, and the common evaluator.

Included arms:
- `RECURRENT_PPO`: the exact learner family frozen for O-R (reference).
- `PPO_DMPLA`: PPO with causal history and David's editable DMPLA/Transformer encoder.
- `DISCRETE_SAC_DMPLA`: categorical SAC for the real four-action space. Standard SB3 SAC is intentionally not used because it requires continuous actions.
- `CUSTOM`: replace the encoder and training loop in the marked cell.

> Everything here is development-only (`949...` tapes). A good result selects a candidate for a later preregistered experiment; it is not Paper 2 evidence.


In [ ]:
# 0) EDIT THIS CONFIG CELL
RUN_PROFILE = 'smoke'            # 'smoke' or 'development'
MODEL_KIND = 'PPO_DMPLA'         # RECURRENT_PPO | PPO_DMPLA | DISCRETE_SAC_DMPLA | CUSTOM
GIT_URL = 'https://github.com/Thom-320/scres-ia.git'
GIT_BRANCH = 'codex/program-o-david-model-workbench'
HISTORY_LEN = 8
POSITIONAL_MODE = 'learned'      # learned | sinusoidal | none
FEATURES_DIM = 128
ATTENTION_HEADS = 4
TRANSFORMER_LAYERS = 2
OPTIMIZER_SEED = 9491
DEVICE = 'auto'

if RUN_PROFILE == 'smoke':
    TOTAL_STEPS, PPO_N_STEPS = 64, 64
    TRAIN_TAPE_RANGE = (949100001, 949100128)
    EVAL_TAPES = list(range(949200001, 949200004))
else:
    TOTAL_STEPS, PPO_N_STEPS = 50_000, 512
    TRAIN_TAPE_RANGE = (949100001, 949119999)
    EVAL_TAPES = list(range(949200001, 949200049))
print(MODEL_KIND, RUN_PROFILE, TOTAL_STEPS, len(EVAL_TAPES), 'evaluation tapes')


In [ ]:
# 1) Portable setup: local, Colab, or Kaggle
from pathlib import Path
import os, subprocess, sys

def run(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.check_call(list(map(str, command)), cwd=cwd)

cwd = Path.cwd().resolve()
if (cwd / 'supply_chain').exists():
    ROOT = cwd
elif Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working/scres-ia')
elif Path('/content').exists():
    ROOT = Path('/content/scres-ia')
else:
    raise RuntimeError('Run from the repo, Colab, or Kaggle')

if not (ROOT / '.git').exists():
    run(['git', 'clone', '--depth', '1', '--branch', GIT_BRANCH, GIT_URL, ROOT])
elif ROOT != cwd:
    run(['git', 'fetch', 'origin', GIT_BRANCH], cwd=ROOT)
    run(['git', 'checkout', GIT_BRANCH], cwd=ROOT)

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
try:
    import gymnasium, stable_baselines3, sb3_contrib, simpy
except ImportError:
    run([sys.executable, '-m', 'pip', 'install', '-q', 'gymnasium>=1.2', 'stable-baselines3>=2.7', 'sb3-contrib>=2.7', 'simpy>=4.1', 'pandas>=2.0', 'matplotlib>=3.7'])
print('ROOT =', ROOT)
print('commit =', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip())


In [ ]:
# 2) Imports, contract, and hard custody checks
import json, math, time
import numpy as np
import pandas as pd
import torch
from torch import nn
from stable_baselines3 import PPO
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from sb3_contrib import RecurrentPPO

from research.program_o_david_workbench import (
    assert_development_seed, compact_summary, evaluate_policy,
    evaluate_policy_against_full_frontiers, integrity_report,
    make_development_env, save_experiment_bundle, swap_product_channels,
)
from research.program_o_discrete_sac import (
    CategoricalSAC, DavidDMPLAEncoder, HistoryPolicyAdapter, HistoryStackWrapper,
)

contract = json.loads((ROOT / 'contracts/program_o_david_model_workbench_v1.json').read_text())
for seed in [*TRAIN_TAPE_RANGE, *EVAL_TAPES]:
    assert_development_seed(seed)
assert MODEL_KIND in {'RECURRENT_PPO', 'PPO_DMPLA', 'DISCRETE_SAC_DMPLA', 'CUSTOM'}
print(contract['status'])
print(contract['physics']['action_space'])
print('Primary metric:', contract['important_metrics']['primary'])


In [ ]:
# 3) Environment smoke: the physics and information boundary are shared by every model
raw_env = make_development_env(root=ROOT, tape_seed_start=TRAIN_TAPE_RANGE[0], tape_seed_end=TRAIN_TAPE_RANGE[1])
obs, info = raw_env.reset(options={'tape_seed': TRAIN_TAPE_RANGE[0], 'cell_index': 0})
print('observation:', obs.shape, obs.dtype, 'range=', (float(obs.min()), float(obs.max())))
print('action space:', raw_env.action_space, 'meaning: 0..3 of next three batch rights go to P_C')
print('observation digest:', info['observation_sha256'])
history_env = HistoryStackWrapper(make_development_env(root=ROOT, tape_seed_start=TRAIN_TAPE_RANGE[0], tape_seed_end=TRAIN_TAPE_RANGE[1]), HISTORY_LEN)
history_obs, _ = history_env.reset()
print('causal stacked history:', history_obs.shape)


## 4) DAVID'S EDITABLE ARCHITECTURE CELL

David can replace the class below completely: add positional encodings, recurrent blocks, attention, convolution, DMPLA/DMLPA variants, or another encoder. Keep `output_dim` and accept a flattened causal history. Do not add tape IDs, future demand, true cell parameters, or latent regime labels.


In [ ]:
# ===== DAVID: EDIT FREELY INSIDE THIS CELL =====
class DavidLabEncoder(DavidDMPLAEncoder):
    def __init__(self, input_dim: int):
        super().__init__(
            input_dim, obs_dim=21, history_len=HISTORY_LEN,
            features_dim=FEATURES_DIM, heads=ATTENTION_HEADS,
            layers=TRANSFORMER_LAYERS, positional_mode=POSITIONAL_MODE,
        )

class DavidSB3Extractor(BaseFeaturesExtractor):
    def __init__(self, observation_space, features_dim=FEATURES_DIM):
        super().__init__(observation_space, features_dim)
        self.encoder = DavidLabEncoder(int(np.prod(observation_space.shape)))
    def forward(self, observations):
        return self.encoder(observations)

# CUSTOM defaults to PPO+your encoder so it runs immediately. Replace this
# function and train_custom_model if you want a completely different algorithm.
def build_custom_model(env):
    return PPO('MlpPolicy', env, seed=OPTIMIZER_SEED, device=DEVICE,
               learning_rate=3e-4, n_steps=PPO_N_STEPS, batch_size=64,
               gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
               policy_kwargs={'features_extractor_class': DavidSB3Extractor,
                              'net_arch': {'pi': [64, 64], 'vf': [64, 64]}},
               verbose=0)

def train_custom_model(model, env, total_steps):
    model.learn(total_timesteps=total_steps, progress_bar=False)
    return model

shape_test = DavidLabEncoder(21 * HISTORY_LEN)(torch.zeros(2, 21 * HISTORY_LEN))
print('David encoder output:', tuple(shape_test.shape))


In [ ]:
# 5) Build exactly one selected model
train_raw = make_development_env(root=ROOT, tape_seed_start=TRAIN_TAPE_RANGE[0], tape_seed_end=TRAIN_TAPE_RANGE[1])
model_label = MODEL_KIND.lower()

if MODEL_KIND == 'RECURRENT_PPO':
    train_env = train_raw
    model = RecurrentPPO(
        'MlpLstmPolicy', train_env, seed=OPTIMIZER_SEED, device=DEVICE,
        learning_rate=3e-4, n_steps=PPO_N_STEPS, batch_size=64, gamma=0.99,
        gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
        policy_kwargs={'lstm_hidden_size': 64, 'net_arch': [64, 64]}, verbose=0)
elif MODEL_KIND in {'PPO_DMPLA', 'CUSTOM'}:
    train_env = HistoryStackWrapper(train_raw, HISTORY_LEN)
    model = build_custom_model(train_env)
elif MODEL_KIND == 'DISCRETE_SAC_DMPLA':
    train_env = HistoryStackWrapper(train_raw, HISTORY_LEN)
    model = CategoricalSAC(
        observation_dim=21 * HISTORY_LEN, action_dim=4, seed=OPTIMIZER_SEED,
        device=DEVICE, encoder_factory=lambda: DavidLabEncoder(21 * HISTORY_LEN))
else:
    raise ValueError(MODEL_KIND)
print(type(model).__name__, 'parameters=', sum(p.numel() for p in (model.policy if hasattr(model, 'policy') else model.actor).parameters()))


In [ ]:
# 6) Train. Smoke proves wiring only; development compares candidate architectures.
started = time.time()
if MODEL_KIND == 'DISCRETE_SAC_DMPLA':
    training_summary = model.learn(
        train_env, total_steps=TOTAL_STEPS, learning_starts=min(32, TOTAL_STEPS // 2),
        batch_size=min(32, max(8, TOTAL_STEPS // 2)), replay_capacity=max(256, TOTAL_STEPS * 2))
elif MODEL_KIND == 'CUSTOM':
    model = train_custom_model(model, train_env, TOTAL_STEPS)
    training_summary = {'executed_steps': int(getattr(model, 'num_timesteps', TOTAL_STEPS))}
else:
    model.learn(total_timesteps=TOTAL_STEPS, progress_bar=False)
    training_summary = {'executed_steps': int(model.num_timesteps)}
elapsed = time.time() - started
print(training_summary)
print(f'training seconds: {elapsed:.2f}')


In [ ]:
# 7) Common adapters. Evaluation always replays the same raw Program O-R tapes.
class RecurrentPolicyAdapter:
    def __init__(self, model, label):
        self.model, self.label = model, label
        self.reset_policy_state()
    def reset_policy_state(self):
        self.state = None
        self.episode_start = np.ones((1,), dtype=bool)
    def predict_action(self, observation):
        action, self.state = self.model.predict(
            observation, state=self.state, episode_start=self.episode_start, deterministic=True)
        self.episode_start[:] = False
        return int(np.asarray(action).item())

if MODEL_KIND == 'RECURRENT_PPO':
    policy = RecurrentPolicyAdapter(model, model_label)
else:
    policy = HistoryPolicyAdapter(model, label=model_label, history_len=HISTORY_LEN)
print('adapter:', type(policy).__name__)


In [ ]:
# 8) Exact mini-evaluator: all three cells, 65,536 open-loop calendars,
# ten classical feedback rules, trajectory audit, and three placebos.
evaluation = evaluate_policy_against_full_frontiers(
    root=ROOT, policy=policy, seeds=EVAL_TAPES, placebo_seed=949299999)
rows = evaluation.policy_rows
scoreboard = evaluation.scoreboard
summary = compact_summary(rows)
display(scoreboard[['model','cell','mean_ret','best_open_loop_ret','best_classical_ret',
                    'H_learned','H_neural','favorable_vs_open_loop',
                    'favorable_vs_classical','unique_calendars','modal_fraction',
                    'feedback_audit_passed','delta_vs_modal','delta_vs_phase_only',
                    'delta_vs_frequency_matched']])
display(scoreboard[['cell','best_open_loop_calendar','best_classical',
                    'ret_full_vs_open_loop','ret_full_vs_classical',
                    'quantity_ret_full_vs_open_loop','quantity_ret_full_vs_classical',
                    'worst_product_fill_vs_open_loop','worst_product_fill_vs_classical',
                    'scheduled_resource_max_spread']])
display(summary[['model','cell','ret_full','quantity_ret_full','worst_product_fill',
                 'lost_orders','unresolved_orders']])
print(json.dumps(integrity_report(rows), indent=2))


In [ ]:
# 9) Fixed-calendar controls on the exact same tapes
class FixedPolicy:
    def __init__(self, action): self.action, self.label = int(action), f'fixed_{action}'
    def reset_policy_state(self): pass
    def predict_action(self, observation): return self.action
baseline_rows = pd.concat([evaluate_policy(root=ROOT, policy=FixedPolicy(a), seeds=EVAL_TAPES) for a in range(4)], ignore_index=True)
baseline_summary = compact_summary(baseline_rows)
display(baseline_summary[['model','cell','ret_visible','ret_full','worst_product_fill']])


In [ ]:
# 10) Minimal feedback audit: diversity and C/H channel swap sensitivity
def first_action(adapter, observation):
    adapter.reset_policy_state()
    return int(adapter.predict_action(observation))
audit_env = make_development_env(root=ROOT, tape_seed_start=EVAL_TAPES[0], tape_seed_end=EVAL_TAPES[-1])
audit_obs, _ = audit_env.reset(options={'tape_seed': EVAL_TAPES[0], 'cell_index': 0})
swapped = swap_product_channels(audit_obs)
print('first action original / C-H swapped:', first_action(policy, audit_obs), first_action(policy, swapped))
print('unique calendars:', rows.calendar.nunique(), 'of', len(rows), '; modal fraction:', rows.calendar.value_counts(normalize=True).iloc[0])
print('A fixed-calendar collapse is not adaptive, even if its mean ReT is high.')


In [ ]:
# 11) Compact plots: only important metrics
import matplotlib.pyplot as plt
plot_rows = pd.concat([rows, baseline_rows], ignore_index=True)
means = plot_rows.groupby(['model','cell'])['ret_visible'].mean().unstack(0)
means.plot(kind='bar', figsize=(11,4), ylabel='canonical ReT', title='Same-tape development comparison')
plt.axhline(0, color='black', linewidth=.8)
plt.tight_layout(); plt.show()


In [ ]:
# 12) Save reproducible development bundle with hashes
from datetime import datetime, timezone
timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
output_dir = ROOT / 'outputs' / 'david_model_lab' / f'{timestamp}_{model_label}'
model_path = output_dir / ('model.pt' if MODEL_KIND == 'DISCRETE_SAC_DMPLA' else 'model.zip')
output_dir.mkdir(parents=True, exist_ok=True)
model.save(str(model_path.with_suffix('') if MODEL_KIND != 'DISCRETE_SAC_DMPLA' else model_path))
if MODEL_KIND != 'DISCRETE_SAC_DMPLA' and not model_path.exists():
    model_path = model_path.with_suffix('.zip')
configuration = {
    'run_profile': RUN_PROFILE, 'model_kind': MODEL_KIND, 'history_len': HISTORY_LEN,
    'positional_mode': POSITIONAL_MODE, 'features_dim': FEATURES_DIM,
    'attention_heads': ATTENTION_HEADS, 'transformer_layers': TRANSFORMER_LAYERS,
    'optimizer_seed': OPTIMIZER_SEED, 'training_tapes': list(TRAIN_TAPE_RANGE),
    'evaluation_tapes': EVAL_TAPES, 'total_steps': TOTAL_STEPS,
}
scoreboard.to_csv(output_dir / 'full_frontier_scoreboard.csv', index=False)
(output_dir / 'full_frontier_diagnostics.json').write_text(json.dumps(evaluation.diagnostics, indent=2, sort_keys=True) + '\n')
manifest = save_experiment_bundle(output_dir=output_dir, rows=rows, summary=summary, model_path=model_path, configuration=configuration)
print(json.dumps(manifest, indent=2))
print('OUTPUT:', output_dir)


## Interpretation rules

1. Compare models on identical `949...` tapes, not training reward.
2. Canonical ReT is primary; full-order ReT, quantity ReT, worst-product fill, lost/unresolved, and resource equality remain visible. CVaR is optional secondary information.
3. Multiple optimizer seeds are needed for a serious architecture comparison. One optimizer seed is only a smoke/debug run.
4. A model that repeats one calendar discovered a schedule; it did not demonstrate feedback.
5. This notebook cannot access scientific `748...` tapes and cannot alter the historical Program O verdict or the frozen O-R run.
6. If David finds a promising model, freeze its code and hyperparameters first, then create a separate preregistered confirmation contract.
